# XNOR-Net — ImageNet Classification Using Binary Convolutional Neural Networks (2016)

_Papers · Efficiency & Compression_

**Replace weights with their signs, but scale by the mean magnitude alpha = mean(|W|) — and convolution becomes XNOR plus popcount.**

---

This notebook is a practice scaffold for the **XNOR-Net — ImageNet Classification Using Binary Convolutional Neural Networks (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- 0. Sanity-check the worked example: alpha = mean(|W|), error with vs without scale. ---
W6 = torch.tensor([0.8, -0.5, 0.3, -0.9, 0.2, -0.4])
B6 = torch.sign(W6)
alpha6 = W6.abs().mean()                       # Eqn. 6
err_scaled = ((W6 - alpha6 * B6) ** 2).sum()   # ||W - alpha*sign(W)||^2
err_naive  = ((W6 - B6) ** 2).sum()            # ||W - sign(W)||^2  (alpha = 1)
print("worked example:  alpha =", round(alpha6.item(), 4),
      " err(scaled) =", round(err_scaled.item(), 4),
      " err(naive) =", round(err_naive.item(), 4))
# worked example:  alpha = 0.5167  err(scaled) = 0.3883  err(naive) = 1.79


# --- 1. Binary-weight convolution as an nn.Module (Eqns 4 and 6). ---
class BinaryWeightConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, stride=1, padding=1, binarize=True):
        super().__init__()
        self.weight   = nn.Parameter(torch.randn(out_ch, in_ch, k, k) * 0.1)
        self.stride   = stride
        self.padding  = padding
        self.binarize = binarize

    def forward(self, x):
        W = self.weight
        if not self.binarize:                          # full-precision baseline
            return F.conv2d(x, W, stride=self.stride, padding=self.padding)
        alpha = W.abs().mean(dim=(1, 2, 3), keepdim=True)   # Eqn. 6: one scale per filter
        B     = torch.sign(W)                               # Eqn. 4: the +/-1 filter
        out   = F.conv2d(x, B, stride=self.stride, padding=self.padding)
        return out * alpha.view(1, -1, 1, 1)                # alpha * (I * B)


# --- 2. Approximation error of a real conv weight tensor: which binarization is closest? ---
W = torch.randn(8, 4, 3, 3)                     # a real filter bank: 8 out, 4 in, 3x3
def rel_err(approx):                            # ||W - approx|| / ||W||
    return (torch.norm(W - approx) / torch.norm(W)).item()

sign_only  = torch.sign(W)                                          # alpha = 1
global_a   = W.abs().mean() * torch.sign(W)                         # one scale for whole tensor
alpha_f    = W.abs().mean(dim=(1, 2, 3), keepdim=True)              # per-filter (the paper)
per_filter = alpha_f * torch.sign(W)

print("rel err  sign() only       :", round(rel_err(sign_only),  4))   # ~0.6704
print("rel err  global alpha      :", round(rel_err(global_a),   4))   # ~0.6104
print("rel err  per-filter alpha  :", round(rel_err(per_filter), 4))   # ~0.6006
# The scaled versions beat naive sign(); per-filter alpha = mean(|W|) is best.
# (Our small run, not the paper's reported number.)


# --- 3. XNOR + popcount == dot product of two sign vectors. ---
torch.manual_seed(1)
x = torch.sign(torch.randn(64))                 # +1 / -1
w = torch.sign(torch.randn(64))
dot = (x * w).sum().item()                       # the normal way
xb, wb  = (x > 0).int(), (w > 0).int()           # encode +1 -> 1, -1 -> 0
popcount = (xb == wb).int().sum().item()         # XNOR then popcount
n = x.numel()
print("dot =", dot, " | 2*popcount - n =", 2 * popcount - n,
      " (popcount =", popcount, ", n =", n, ")")
# dot = 0.0  | 2*popcount - n = 0   ->  the bitwise route matches exactly.

## Visualize the data & results

_Approximating a real conv weight tensor with 1 bit per weight: how much does the scale alpha = mean(|W|) reduce the error versus naive sign()?_

In [ ]:
import torch

# Approximation error of a real weight tensor under three 1-bit binarizations.
# Reproduces the qualitative effect: the scale alpha = mean(|W|) beats naive sign().
torch.manual_seed(0)
W = torch.randn(8, 4, 3, 3)                       # a real conv filter bank

def rel_err(approx):
    return (torch.norm(W - approx) / torch.norm(W)).item()

sign_only  = torch.sign(W)                         # alpha = 1 (no scale)
global_a   = W.abs().mean() * torch.sign(W)        # one scale for the whole tensor
alpha_f    = W.abs().mean(dim=(1, 2, 3), keepdim=True)   # Eqn. 6, per output filter
per_filter = alpha_f * torch.sign(W)

print("sign() only       :", round(rel_err(sign_only),  4))   # 0.6704
print("one global alpha  :", round(rel_err(global_a),   4))   # 0.6104
print("per-filter alpha  :", round(rel_err(per_filter), 4))   # 0.6006
# Lower is better. The mean-magnitude scale recovers error that plain sign() lost.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a real conv weight tensor. Approximate it as $\alpha\,\mathrm{sign}(W)$
            with $\alpha=\mathrm{mean}(|W|)$, and also as plain $\mathrm{sign}(W)$ (set $\alpha=1$). Compare the
            reconstruction error. What does the gap show?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the per-filter scale alpha = W.abs().mean(dim=(1,2,3), keepdim=True) (Eqn. 6) and the binary filter B = torch.sign(W). — _These are the paper's optimal $\alpha$ and $B$ &mdash; the scaled binary copy $\alpha B$ of the real filter._
- Measure relative error norm(W - alpha*B)/norm(W), then again with alpha=1. — _The only change is the scale, so any difference is attributable to $\alpha$ alone &mdash; an honest ablation._
- Observe the scaled version's error is smaller (our run: ~0.60 vs ~0.67). — _The mean-magnitude scale rebuilds the filter's overall size that plain sign threw away._

**Answer:** The scaled binary filter has lower reconstruction error than plain $\mathrm{sign}(W)$ (our
                 small run: relative error ~0.60 with the scale vs ~0.67 without). Since the only difference is
                 $\alpha=\mathrm{mean}(|W|)$, this isolates the scale as the reason binary weights approximate the
                 real filter well &mdash; exactly the paper's claim. It is one number per filter, and it is the
                 provably optimal one.

</details>

**Problem 2.** Two sign vectors $x=[+1,-1,-1,+1]$ and $w=[+1,+1,-1,-1]$. Compute their dot product the normal way,
            then via XNOR and popcount, and show they agree.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Normal dot product: $x^{\top}w = (1)(1)+(-1)(1)+(-1)(-1)+(1)(-1) = 1-1+1-1 = 0$. — _A $\pm1$ dot product counts agreements minus disagreements; here 2 agree, 2 disagree, net 0._
- Encode $+1\to1$, $-1\to0$: $x_b=[1,0,0,1]$, $w_b=[1,1,0,0]$. XNOR (equal bits): $[1,0,1,0]$, popcount $=2$. — _XNOR returns 1 exactly where the signs agree; popcount counts those agreements ($p=2$)._
- Recover the dot product: $2p - n = 2\cdot2 - 4 = 0$. Matches. — _With $p$ agreements out of $n$, agreements minus disagreements is $p-(n-p)=2p-n$._

**Answer:** Both routes give $0$. The bitwise route &mdash; XNOR then popcount then $2p-n$ &mdash; reproduces
                 the floating-point dot product exactly, with no multiplications. That is why XNOR-Networks can do
                 the convolution's multiply-accumulate as bit operations (&sect;3.2).

</details>

**Problem 3.** Your worked example had $W=[0.8,-0.5,0.3,-0.9,0.2,-0.4]$ with $\alpha=0.5167$ giving squared error
            $0.388$, while naive sign gave $1.79$. Suppose someone "improves" things by using $\alpha=1.0$
            instead. Does the error go up or down, and why is $\mathrm{mean}(|W|)$ special?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute error at $\alpha=1$: $\lVert W-\mathrm{sign}(W)\rVert^2 = 1.79$ (the naive case). — _$\alpha=1$ is exactly plain sign with no rescaling._
- Recall $J(\alpha)=n\alpha^2-2\alpha\sum|W_i|+\sum W_i^2$ is an upward parabola minimized at $\alpha=\frac{1}{n}\sum|W_i|$. — _Eqn. 6: any $\alpha$ other than the mean magnitude gives a strictly larger error._
- So $\alpha=1$ (here $1.79$) is worse than $\alpha=0.5167$ (here $0.388$). — _$0.5167$ sits at the bottom of the parabola; moving away to $1.0$ climbs back up the curve._

**Answer:** The error goes up at $\alpha=1.0$ (back to $1.79$). The error as a function of $\alpha$
                 is an upward parabola whose minimum is at $\alpha=\mathrm{mean}(|W|)=0.5167$ (Eqn. 6). Any other
                 scale &mdash; including $1.0$ &mdash; is strictly worse. That is what "optimal" means here: not just
                 a heuristic, but the exact minimizer of the reconstruction error.

</details>